# 🚀 Project Setup — Educational Quickstart Blueprint

**AI Learning Playground — Environment Configuration & Model Download**

Run this notebook **once** before opening any of the four starter notebooks.
It installs all dependencies, validates your GPU, authenticates with Hugging Face,
and downloads the quantized GGUF models used across all four blueprints.

---

## What This Notebook Does

| Cell | Section | Purpose |
|------|---------|---------|
| 2 | CUDA Configuration | Sets environment variables — **must run before any torch import** |
| 3 | torchvision/torchaudio Install | Installs PyTorch vision & audio libraries with CUDA 12.8 |
| 4 | PyTorch Validation | Confirms pre-installed PyTorch + CUDA are working |
| 6 | Library Imports | Imports core libraries; prints system info |
| 8 | GPU Validation | 4-step test: detection → alloc → matmul → cleanup |
| 10 | System Dependencies | Installs `ffmpeg`, `portaudio19-dev` via `apt-get` |
| 12 | AI Library Install | Installs `transformers`, `diffusers`, `mlflow`, `streamlit`, etc. (3–7 min) |
| 14 | Infrastructure Test | Verifies all library imports; prints version numbers; pass/fail summary |
| 16 | Hugging Face Auth | Guided token entry; saves credentials for gated-model downloads |
| 18 | Model Download | Downloads all 5 GGUF models to `/home/jovyan/local/` |
| 20 | Setup Summary | GPU info, auth status, model paths, next steps |
| 22 | Quick Reference | Code snippets for Zephyr, FLUX, Whisper, XTTS |

## After Setup — Starter Notebooks

| Notebook | Capability | Model Used |
|----------|-----------|------------|
| [chatbot-starter.ipynb](chatbot-starter.ipynb) | Conversational AI | Zephyr 7B Beta Q5_K_M |
| [document-analyzer-starter.ipynb](document-analyzer-starter.ipynb) | Document Q&A (RAG) | Llama 3.1 8B Q6_K_L |
| [image-gen-starter.ipynb](image-gen-starter.ipynb) | Text-to-Image | FLUX.1-dev Q4_K_S |
| [voice-assistant-starter.ipynb](voice-assistant-starter.ipynb) | Voice AI (STT + TTS) | Whisper V3 Turbo + XTTS v2 |

## 1. Environment Configuration

Set the required environment variables **before any `import torch` call**.
CUDA variables are read by the runtime at import time — they cannot be changed after the first GPU import.

### CUDA Variables
| Variable | Value | Why |
|----------|-------|-----|
| `CUDA_VISIBLE_DEVICES` | `"0"` | Restricts PyTorch to GPU 0; avoids accidental multi-GPU fragmentation |
| `PYTORCH_CUDA_ALLOC_CONF` | `"expandable_segments:True"` | Reduces memory fragmentation by letting the allocator grow segments dynamically |
| `CUDA_LAUNCH_BLOCKING` | `"0"` | Async kernel launches (set to `"1"` only when debugging CUDA errors) |

### Hugging Face Cache
| Variable | Value | Purpose |
|----------|-------|---------|
| `HF_HOME` | `local/data/huggingface` | Centralized Hugging Face cache — survives workspace restarts |
| `TRANSFORMERS_CACHE` | `local/data/huggingface/hub` | Transformer model cache directory |

> ⚠️ **Run this cell first** — before importing `torch` or any other GPU/AI library.

In [ ]:
import os
import sys
import time
import logging
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
logging.getLogger("mlflow").setLevel(logging.ERROR)
logging.getLogger("transformers").setLevel(logging.ERROR)

HF_HOME = Path("/home/jovyan/local/data/huggingface")
TRANSFORMERS_CACHE = Path("/home/jovyan/local/data/hugginface/hub")

# ── CUDA environment — must be set before any torch import ───────────────────

# Restrict execution to GPU device 0 (avoids accidental multi-GPU problems)
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

# Allow PyTorch's memory allocator to grow segments dynamically.
# This significantly reduces OOM errors when loading large models.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# Keep CUDA launches asynchronous for performance.
# Set to "1" only when debugging CUDA errors (shows exact failing line).
os.environ.setdefault("CUDA_LAUNCH_BLOCKING", "0")

# ── Hugging Face cache — persists across workspace restarts ──────────────────
os.environ.setdefault("HF_HOME", str(HF_HOME))
os.environ.setdefault("TRANSFORMERS_CACHE", str(TRANSFORMERS_CACHE))

# ── Ensure project root is on the Python path so `src.*` imports work ─────────
sys.path.insert(0, "..")

start_time = time.time()

print("─" * 55)
print("  Environment Configuration")
print("─" * 55)
print("  CUDA:")
print(f"    CUDA_VISIBLE_DEVICES     : {os.environ['CUDA_VISIBLE_DEVICES']}")
print(f"    PYTORCH_CUDA_ALLOC_CONF  : {os.environ['PYTORCH_CUDA_ALLOC_CONF']}")
print(f"    CUDA_LAUNCH_BLOCKING     : {os.environ['CUDA_LAUNCH_BLOCKING']}")
print()
print("  Hugging Face Cache:")
print(f"    HF_HOME                  : {os.environ['HF_HOME']}")
print(f"    TRANSFORMERS_CACHE       : {os.environ['TRANSFORMERS_CACHE']}")
print("─" * 55)
print("✅ Cell 1 complete — environment variables configured")
print("⏱️  Setup notebook started")


───────────────────────────────────────────────────────
  Environment Configuration
───────────────────────────────────────────────────────
  CUDA:
    CUDA_VISIBLE_DEVICES     : 0
    PYTORCH_CUDA_ALLOC_CONF  : expandable_segments:True
    CUDA_LAUNCH_BLOCKING     : 0

  Hugging Face Cache:
    HF_HOME                  : /home/jovyan/local/data/huggingface
    TRANSFORMERS_CACHE       : /home/jovyan/local/data/hugginface/hub
───────────────────────────────────────────────────────
✅ Cell 1 complete — environment variables configured
⏱️  Setup notebook started


## 1.5 PyTorch Vision & Audio Libraries

Install torchvision and torchaudio with CUDA 12.8 support (cu128).

These libraries extend PyTorch with computer vision and audio processing capabilities.
They must be installed from the official PyTorch CUDA 12.8 wheels — they are not available on PyPI.

| Library | Purpose |
|---------|---------|
| `torchvision` | Computer vision ops (FLUX image generation) |
| `torchaudio` | Audio I/O and signal processing |

> **Note:** The voice assistant's STT loads Whisper via `AutoModelForSpeechSeq2Seq` + `AutoProcessor` directly (from `transformers`, installed in Cell 6). This bypasses `torchaudio`/`torchcodec` entirely — `WhisperFeatureExtractor` computes mel spectrograms with pure numpy/scipy.

> **First-run time:** ~1-2 minutes (skipped if already installed).

In [2]:
import sys
sys.path.insert(0, "..")
from src.utils import pip_install

print("─" * 55)
print("  torchvision + torchaudio (CUDA 12.8 / cu128)")
print("─" * 55)

_tv_ok = False
try:
    import torchvision, torchaudio
    print(f"  ✅ torchvision  : {torchvision.__version__} (already installed)")
    print(f"  ✅ torchaudio   : {torchaudio.__version__} (already installed)")
    _tv_ok = True
except ImportError:
    print("  ⬇️  torchvision/torchaudio not found — installing...")
    _rc, _err = pip_install(
        "torchvision==0.24.1", "torchaudio==2.9.1",
        "--index-url", "https://download.pytorch.org/whl/cu128",
    )
    if _rc == 0:
        # Clear only the newly installed packages — do NOT clear torch itself.
        # Removing torch from sys.modules while the C extension is already loaded
        # causes a RuntimeError when torchvision tries to re-import it.
        for _mod in list(sys.modules.keys()):
            if any(_mod.startswith(p) for p in ("torchvision", "torchaudio")):
                del sys.modules[_mod]
        try:
            import torchvision, torchaudio  # noqa: F811
            print(f"  ✅ torchvision  : {torchvision.__version__}")
            print(f"  ✅ torchaudio   : {torchaudio.__version__}")
            _tv_ok = True
        except Exception as e:
            print(f"  ❌ Installed but import failed: {str(e)[:80]}")
            print("       Please restart the kernel and re-run from Cell 1.")
            print("─" * 55)
            raise RuntimeError("❌ Kernel restart required after first-time torchvision/torchaudio install")
    else:
        print(f"  ❌ install failed (rc={_rc})")
        print(f"       {_err[-200:].strip()}")
        print("─" * 55)
        raise RuntimeError("❌ torchvision/torchaudio installation failed")

print("─" * 55)
print("✅ Cell 1.5 complete — torchvision/torchaudio installed")

───────────────────────────────────────────────────────
  torchvision + torchaudio (CUDA 12.8 / cu128)
───────────────────────────────────────────────────────
  ⬇️  torchvision/torchaudio not found — installing...
  ✅ torchvision  : 0.24.1+cu128
  ✅ torchaudio   : 2.9.1+cu128
───────────────────────────────────────────────────────
✅ Cell 1.5 complete — torchvision/torchaudio installed


## 2. PyTorch Validation

Confirm that the pre-installed PyTorch build has CUDA support.

In [3]:
import torch

cuda_ok = torch.cuda.is_available()

print("─" * 55)
print("  PyTorch Validation (pre-installed)")
print("─" * 55)
print(f"  PyTorch  : {torch.__version__}")
print(f"  CUDA     : {'✅ Available' if cuda_ok else '❌ Not found — GPU required'}")

if cuda_ok:
    props = torch.cuda.get_device_properties(0)
    vram_gb = props.total_memory / 1e9
    print(f"  GPU      : {props.name}")
    print(
        f"  VRAM     : {vram_gb:.1f} GB {'✅' if vram_gb >= 8 else '⚠️ (8 GB minimum recommended)'}"
    )
    print(f"  Compute  : {props.major}.{props.minor}")

    # Minimum version gate — torch >= 2.8.0 required by this project
    parts = torch.__version__.split("+")[0].split(".")
    if int(parts[0]) < 2 or (int(parts[0]) == 2 and int(parts[1]) < 8):
        print(
            f"  ⚠️  torch {torch.__version__} is below minimum — this project requires torch ≥ 2.8.0"
        )
    else:
        print(f"  ✅ torch {torch.__version__} meets minimum requirement (≥ 2.8.0)")
else:
    print(
        "  ⚠️ GPU not detected. All models in this project require a CUDA-capable GPU."
    )

print("─" * 55)
print(f"✅ Cell 2 complete — PyTorch {torch.__version__} verified")

───────────────────────────────────────────────────────
  PyTorch Validation (pre-installed)
───────────────────────────────────────────────────────
  PyTorch  : 2.9.1+cu128
  CUDA     : ✅ Available
  GPU      : NVIDIA RTX PRO 5000 Blackwell Generation Laptop GPU
  VRAM     : 25.7 GB ✅
  Compute  : 12.0
  ✅ torch 2.9.1+cu128 meets minimum requirement (≥ 2.8.0)
───────────────────────────────────────────────────────
✅ Cell 2 complete — PyTorch 2.9.1+cu128 verified


## 3. Library Imports

Confirm that the core Python standard library and PyTorch are importable and print
the runtime environment details.
This snapshot helps reproduce bugs: paste the output when asking for support.

In [4]:
import os
import sys
import platform
import torch

print("─" * 55)
print("  System Information")
print("─" * 55)
print(f"  Python   : {sys.version.split()[0]}")
print(f"  Platform : {platform.system()} {platform.release()}")
print(f"  Arch     : {platform.machine()}")
print()
print("  Core PyTorch")
print(f"  torch    : {torch.__version__}")
print(f"  CUDA     : {torch.version.cuda or 'None'}")
print(f"  cuDNN    : {torch.backends.cudnn.version() or 'None'}")
print("─" * 55)
print("✅ Cell 3 complete — core imports confirmed")

───────────────────────────────────────────────────────
  System Information
───────────────────────────────────────────────────────
  Python   : 3.12.7
  Platform : Linux 5.15.167.4-microsoft-standard-WSL2
  Arch     : x86_64

  Core PyTorch
  torch    : 2.9.1+cu128
  CUDA     : 12.8
  cuDNN    : 91002
───────────────────────────────────────────────────────
✅ Cell 3 complete — core imports confirmed


## 4. GPU Validation

Run a 4-step functional test to confirm your GPU is correctly bound to PyTorch and can
perform the operations that the AI models rely on.

| Step | Test | What it checks |
|------|------|---------------|
| 1 | CUDA detection | `torch.cuda.is_available()` returns `True` |
| 2 | Memory allocation | Can allocate a 4 MB tensor on GPU memory |
| 3 | Matrix multiply | Can run a 512×512 fp32 matmul on GPU |
| 4 | VRAM cleanup | `torch.cuda.empty_cache()` + `gc.collect()` succeed |

All 4 steps must pass before proceeding to model downloads.

In [5]:
import gc
import torch

results = []

print("─" * 55)
print("  GPU Validation — 4-step functional test")
print("─" * 55)

# ── Step 1: CUDA detection ────────────────────────────────────────────────────
cuda_ok = torch.cuda.is_available()
results.append(("CUDA detection", cuda_ok,
                f"Device: {torch.cuda.get_device_name(0)}" if cuda_ok else "CUDA not found"))
print(f"[{'✅ PASS' if cuda_ok else '❌ FAIL'}] Step 1 — CUDA detection"
      + (f"  ({torch.cuda.get_device_name(0)})" if cuda_ok else ""))

# ── Step 2: Memory allocation ─────────────────────────────────────────────────
alloc_ok = False
alloc_tensor = None
if cuda_ok:
    try:
        # Allocate a ~4 MB float32 tensor on the GPU (1024 × 1024 × 4 bytes)
        alloc_tensor = torch.empty(1024, 1024, dtype=torch.float32, device="cuda")
        alloc_ok = alloc_tensor.is_cuda
        results.append(("Memory allocation", alloc_ok, "4 MB tensor allocated on GPU"))
        print(f"[{'✅ PASS' if alloc_ok else '❌ FAIL'}] Step 2 — Memory allocation  (4 MB tensor on GPU)")
    except Exception as e:
        results.append(("Memory allocation", False, str(e)))
        print(f"[❌ FAIL] Step 2 — Memory allocation - try shutting down the kernel and start again ({e})")
else:
    results.append(("Memory allocation", False, "Skipped — no CUDA"))
    print("[⏭  SKIP] Step 2 — Memory allocation  (no CUDA)")

# ── Step 3: Matrix operation ──────────────────────────────────────────────────
matmul_ok = False
if alloc_ok:
    try:
        a = torch.randn(512, 512, dtype=torch.float32, device="cuda")
        b = torch.randn(512, 512, dtype=torch.float32, device="cuda")
        c = torch.matmul(a, b)
        matmul_ok = c.shape == (512, 512)
        results.append(("Matrix operation", matmul_ok, "512×512 fp32 matmul succeeded"))
        print(f"[{'✅ PASS' if matmul_ok else '❌ FAIL'}] Step 3 — Matrix operation   (512×512 fp32 matmul)")
    except Exception as e:
        results.append(("Matrix operation", False, str(e)))
        print(f"[❌ FAIL] Step 3 — Matrix operation   ({e})")
else:
    results.append(("Matrix operation", False, "Skipped — previous step failed"))
    print("[⏭  SKIP] Step 3 — Matrix operation   (previous step failed)")

# ── Step 4: Memory cleanup ────────────────────────────────────────────────────
cleanup_ok = False
try:
    # Release tensors and free the CUDA cache
    for _t in ("alloc_tensor", "a", "b", "c"):
        if _t in dir():
            del locals()[_t]
    torch.cuda.empty_cache()
    gc.collect()
    cleanup_ok = True
    results.append(("Memory cleanup", True, "empty_cache() + gc.collect() succeeded"))
    print(f"[✅ PASS] Step 4 — Memory cleanup      (empty_cache + gc.collect)")
except Exception as e:
    results.append(("Memory cleanup", False, str(e)))
    print(f"[❌ FAIL] Step 4 — Memory cleanup      ({e})")

# ── Summary ───────────────────────────────────────────────────────────────────
passed = sum(1 for _, ok, _ in results if ok)
total  = len(results)
print("─" * 55)
if passed == total:
    print(f"✅ Cell 4 complete — all {total}/{total} steps passed")
else:
    print(f"⚠️  Cell 4 complete — {passed}/{total} steps passed")
    for name, ok, detail in results:
        if not ok:
            print(f"   ❌ {name}: {detail}")


───────────────────────────────────────────────────────
  GPU Validation — 4-step functional test
───────────────────────────────────────────────────────
[✅ PASS] Step 1 — CUDA detection  (NVIDIA RTX PRO 5000 Blackwell Generation Laptop GPU)
[✅ PASS] Step 2 — Memory allocation  (4 MB tensor on GPU)
[✅ PASS] Step 3 — Matrix operation   (512×512 fp32 matmul)
[✅ PASS] Step 4 — Memory cleanup      (empty_cache + gc.collect)
───────────────────────────────────────────────────────
✅ Cell 4 complete — all 4/4 steps passed


## 5. System Dependencies

Install the system-level packages required by the AI libraries in this blueprint.
These are **OS packages** installed via `apt-get`, not Python packages.

| Package | Purpose | Required by |
|---------|---------|-------------|
| `ffmpeg` | Audio/video transcoding — decodes MP3, OGG, FLAC → WAV | Whisper (transformers), XTTS v2 (CoquiTTS) |
| `portaudio19-dev` | System audio driver for real-time capture/playback | `pyaudio` (Python binding for PortAudio) |
| `git-lfs` | Git Large File Storage for handling large model files | Hugging Face Hub, model repositories |

> **First-run time:** ~30 seconds.

In [6]:
import subprocess

print("─" * 55)
print("  System Dependencies Install")
print("─" * 55)

# Install ffmpeg (audio transcoding for Whisper/TTS), portaudio19-dev (PortAudio C headers for pyaudio),
# and git-lfs (Git Large File Storage for handling large model files).
try:
    subprocess.run(["sudo", "apt-get", "update", "-qq"], check=True, capture_output=True)
    subprocess.run(
        ["sudo", "apt-get", "install", "-y", "-qq", "ffmpeg", "portaudio19-dev", "git-lfs"],
        check=True, capture_output=True
    )
    print("  ✅ Installed: ffmpeg, portaudio19-dev, git-lfs")
except Exception as e:
    print(f"  ⚠️  apt-get failed: {e}")
    print("       ffmpeg is required for audio processing (Whisper STT, XTTS TTS).")
    print("       portaudio19-dev is required by pyaudio (real-time microphone capture).")
    print("       git-lfs is required for handling large model files.")
    print("       If sudo is unavailable, these may already be present in the base image.")

# Verify ffmpeg is accessible
try:
    out = subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True)
    first_line = out.stdout.strip().split("\n")[0]
    print(f"  ✅ ffmpeg   : {first_line}")
except Exception:
    print("  ❌ ffmpeg   : not found — audio processing will be unavailable")

# Verify git-lfs is accessible
try:
    out = subprocess.run(["git-lfs", "--version"], capture_output=True, text=True)
    version_line = out.stdout.strip().split("\n")[0]

    print(f"  ✅ git-lfs  : {version_line}")

except Exception:
    print("  ❌ git-lfs  : not found — large file downloads may fail")

print("─" * 55)
print("✅ Cell 5 complete — system dependencies installed")


───────────────────────────────────────────────────────
  System Dependencies Install
───────────────────────────────────────────────────────
  ⚠️  apt-get failed: Command '['sudo', 'apt-get', 'install', '-y', '-qq', 'ffmpeg', 'portaudio19-dev', 'git-lfs']' returned non-zero exit status 100.
       ffmpeg is required for audio processing (Whisper STT, XTTS TTS).
       portaudio19-dev is required by pyaudio (real-time microphone capture).
       git-lfs is required for handling large model files.
       If sudo is unavailable, these may already be present in the base image.
  ✅ ffmpeg   : ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  ✅ git-lfs  : git-lfs/3.0.2 (GitHub; linux amd64; go 1.18.1)
───────────────────────────────────────────────────────
✅ Cell 5 complete — system dependencies installed


## 6. AI Library Install

Install all project AI dependencies from `requirements.txt`.

Two packages need special handling and are installed separately with fallback logic:

| Package | Issue | Strategy |
|---------|-------|----------|
| `pyaudio` | Requires `portaudio.h` C headers from Cell 5. | Attempted after system deps; failure is non-blocking (no starter notebook uses real-time mic capture). |
| `faiss-gpu-cu12` | No pre-built wheel for CUDA 12.8 + Python 3.12. | GPU install attempted first; automatic fallback to `faiss-cpu` if it fails. |

> **Note:** torchvision and torchaudio were installed in Cell 1.5.

> **First-run time:** 3–7 minutes.

In [7]:
import sys
sys.path.insert(0, "..")
from src.utils import pip_install

# ── Stage 1: Core requirements (fragile packages excluded) ────────────────────
print("─" * 55)
print("  Stage 1: Core requirements")
print("─" * 55)

with open("../requirements.txt") as _f:
    _safe = [
        l.strip() for l in _f
        if l.strip()
        and not l.strip().startswith("#")
        and not l.strip().startswith("--")
        and "faiss-gpu-cu12" not in l
        and "pyaudio" not in l.lower()
        and "coqui-tts" not in l
    ]

_rc, _err = pip_install(*_safe)
if _rc == 0:
    print("  ✅ Core requirements installed")
else:
    print(f"  ⚠️  Some packages had issues (rc={_rc}). Continuing.")
    print(f"       {_err[-200:].strip()}")

# ── Stage 2: pyaudio (requires portaudio19-dev from Cell 5) ──────────────────
print()
print("─" * 55)
print("  Stage 2: pyaudio")
print("─" * 55)
_pyaudio_ok = False
try:
    import pyaudio as _pa  # noqa: F401
    print("  ✅ pyaudio already installed")
    _pyaudio_ok = True
except ImportError:
    _rc, _ = pip_install("pyaudio")
    if _rc == 0:
        print("  ✅ pyaudio installed")
        _pyaudio_ok = True
    else:
        print("  ⚠️  pyaudio failed — real-time mic capture unavailable")
        print("       Ensure Cell 5 ran successfully, then re-run this cell.")
        print("       No starter notebook requires real-time audio (voice demo uses file upload).")

# ── Stage 3: FAISS — GPU preferred, CPU fallback ─────────────────────────────
print()
print("─" * 55)
print("  Stage 3: FAISS (GPU preferred → CPU fallback)")
print("─" * 55)
_faiss_gpu = False
try:
    import faiss as _f
    _faiss_gpu = hasattr(_f, "StandardGpuResources")
    print(f"  ✅ faiss already installed ({'GPU' if _faiss_gpu else 'CPU'})")
except ImportError:
    _rc, _ = pip_install("faiss-gpu-cu12")
    if _rc == 0:
        print("  ✅ faiss-gpu-cu12 installed")
        _faiss_gpu = True
    else:
        print("  ⚠️  faiss-gpu-cu12: no compatible wheel for CUDA 12.8 / Python 3.12")
        _rc2, _ = pip_install("faiss-cpu")
        if _rc2 == 0:
            print("  ✅ faiss-cpu installed (CPU fallback — functionally identical at educational scale)")
        else:
            print("  ❌ faiss-cpu also failed — check pip output above")

# ── Stage 4: coqui-tts[codec] — CoquiTTS / XTTS v2 voice synthesis ───────────
# coqui-tts[codec] is the Idiap Research Institute fork of Coqui TTS, which
# supports Python 3.12.  The [codec] extra adds torchcodec for audio I/O,
# required from PyTorch 2.9+.  Cell 5 already installs ffmpeg; espeak-ng is
# needed for phoneme-based TTS backends.
# Installing separately from Stage 1 gives a clear, targeted error message
# if something goes wrong — students know exactly which component failed.
print()
print("─" * 55)
print("  Stage 4: coqui-tts[codec] (CoquiTTS / XTTS v2 voice synthesis)")
print("─" * 55)
_tts_ok = False
try:
    import TTS as _tts_check  # noqa: F401
    print(f"  ✅ coqui-tts[codec] already installed  ({_tts_check.__version__})")
    _tts_ok = True
except ImportError:
    print("  ⬇️  Installing coqui-tts[codec]...")
    _rc, _err = pip_install("coqui-tts[codec] @ git+https://github.com/idiap/coqui-ai-TTS")
    if _rc == 0:
        try:
            import TTS as _tts_check  # noqa: F811
            print(f"  ✅ coqui-tts[codec] installed  ({_tts_check.__version__})")
            _tts_ok = True
        except Exception as _e:
            print(f"  ❌ coqui-tts[codec] installed but import failed: {str(_e)[:80]}")
            print("       Please restart the kernel and re-run from Cell 1.")
    else:
        print(f"  ❌ coqui-tts[codec] install failed (rc={_rc})")
        print(f"       {_err[-300:].strip()}")
        print()
        print("  Troubleshooting:")
        print("  1. Ensure Cell 5 ran successfully (ffmpeg must be installed).")
        print("  2. Install espeak-ng system dependency manually:")
        print("       !sudo apt-get install -y espeak-ng")
        print("  3. Then re-run this cell.")
        print("  Note: The voice assistant starter notebook requires coqui-tts[codec] for")
        print("        XTTS v2 text-to-speech synthesis.  Other starters are unaffected.")

# ── Summary ───────────────────────────────────────────────────────────────────
print()
print("─" * 55)
print(f"  FAISS           : {'GPU (faiss-gpu-cu12)' if _faiss_gpu else 'CPU fallback (faiss-cpu)'}")
print(f"  pyaudio         : {'installed' if _pyaudio_ok else 'UNAVAILABLE — real-time mic disabled (no starter notebook affected)'}")
print(f"  coqui-tts[codec]: {'installed' if _tts_ok else 'UNAVAILABLE — voice TTS disabled (see troubleshooting above)'}")
print("─" * 55)
print("✅ Cell 6 complete — AI libraries installed")


───────────────────────────────────────────────────────
  Stage 1: Core requirements
───────────────────────────────────────────────────────
  ✅ Core requirements installed

───────────────────────────────────────────────────────
  Stage 2: pyaudio
───────────────────────────────────────────────────────
  ✅ pyaudio installed

───────────────────────────────────────────────────────
  Stage 3: FAISS (GPU preferred → CPU fallback)
───────────────────────────────────────────────────────
  ✅ faiss-gpu-cu12 installed

───────────────────────────────────────────────────────
  Stage 4: coqui-tts[codec] (CoquiTTS / XTTS v2 voice synthesis)
───────────────────────────────────────────────────────
  ⬇️  Installing coqui-tts[codec]...
  ✅ coqui-tts[codec] installed  (0.28.0.dev0)

───────────────────────────────────────────────────────
  FAISS           : GPU (faiss-gpu-cu12)
  pyaudio         : installed
  coqui-tts[codec]: installed
───────────────────────────────────────────────────────
✅ Cell 6

## 7. Infrastructure Test

Import every critical AI library and confirm its version number.
A 5/5 pass means the environment is fully wired and all starter notebooks can run.

In [8]:
total = 0
passed = 0

print("═" * 60)
print("  Infrastructure Test — AI Libraries")
print("═" * 60)

# ── 4.2.1 Core ML/AI Frameworks ──────────────────────────────────────────────
print("\n  4.2.1 — Core ML/AI Frameworks")
print("  " + "─" * 56)

try:
    import transformers
    print(f"  ✅ {'transformers':<32} {transformers.__version__}"); passed += 1
except Exception as e:
    print(f"  ❌ {'transformers':<32} {str(e)[:48]}")
total += 1

try:
    import diffusers
    print(f"  ✅ {'diffusers':<32} {diffusers.__version__}"); passed += 1
except Exception as e:
    print(f"  ❌ {'diffusers':<32} {str(e)[:48]}")
total += 1

try:
    import accelerate
    print(f"  ✅ {'accelerate':<32} {accelerate.__version__}"); passed += 1
except Exception as e:
    print(f"  ❌ {'accelerate':<32} {str(e)[:48]}")
total += 1

try:
    import bitsandbytes
    print(f"  ✅ {'bitsandbytes':<32} {bitsandbytes.__version__}"); passed += 1
except Exception as e:
    print(f"  ❌ {'bitsandbytes':<32} {str(e)[:48]}")
total += 1

try:
    import safetensors
    print(f"  ✅ {'safetensors':<32} {safetensors.__version__}"); passed += 1
except Exception as e:
    print(f"  ❌ {'safetensors':<32} {str(e)[:48]}")
total += 1

try:
    import sentencepiece
    print(f"  ✅ {'sentencepiece':<32} {sentencepiece.__version__}"); passed += 1
except Exception as e:
    print(f"  ❌ {'sentencepiece':<32} {str(e)[:48]}")
total += 1

try:
    import vllm
    print(f"  ✅ {'vllm':<32} {vllm.__version__}"); passed += 1
except Exception as e:
    print(f"  ❌ {'vllm':<32} {str(e)[:48]}")
total += 1

try:
    import xformers
    print(f"  ✅ {'xformers':<32} {xformers.__version__}"); passed += 1
except Exception as e:
    print(f"  ❌ {'xformers':<32} {str(e)[:48]}")
total += 1

# ── 4.2.2 Interface & Deployment ─────────────────────────────────────────────
print("\n  4.2.2 — Interface & Deployment")
print("  " + "─" * 56)

try:
    import mlflow
    print(f"  ✅ {'mlflow':<32} {mlflow.__version__}"); passed += 1
except Exception as e:
    print(f"  ❌ {'mlflow':<32} {str(e)[:48]}")
total += 1

try:
    import huggingface_hub
    print(f"  ✅ {'huggingface_hub':<32} {huggingface_hub.__version__}"); passed += 1
except Exception as e:
    print(f"  ❌ {'huggingface_hub':<32} {str(e)[:48]}")
total += 1

try:
    import fastapi
    print(f"  ✅ {'fastapi':<32} {fastapi.__version__}"); passed += 1
except Exception as e:
    print(f"  ❌ {'fastapi':<32} {str(e)[:48]}")
total += 1

try:
    import pydantic
    print(f"  ✅ {'pydantic':<32} {pydantic.__version__}"); passed += 1
except Exception as e:
    print(f"  ❌ {'pydantic':<32} {str(e)[:48]}")
total += 1

try:
    import datasets
    print(f"  ✅ {'datasets':<32} {datasets.__version__}"); passed += 1
except Exception as e:
    print(f"  ❌ {'datasets':<32} {str(e)[:48]}")
total += 1

try:
    import streamlit
    print(f"  ✅ {'streamlit':<32} {streamlit.__version__}"); passed += 1
except Exception as e:
    print(f"  ❌ {'streamlit':<32} {str(e)[:48]}")
total += 1

# ── 4.2.3 Document & Media Processing ────────────────────────────────────────
print("\n  4.2.3 — Document & Media Processing")
print("  " + "─" * 56)

try:
    import PyPDF2
    print(f"  ✅ {'PyPDF2':<32} {PyPDF2.__version__}"); passed += 1
except Exception as e:
    print(f"  ❌ {'PyPDF2':<32} {str(e)[:48]}")
total += 1

try:
    import fitz
    print(f"  ✅ {'pymupdf (fitz)':<32} {fitz.__version__}"); passed += 1
except Exception as e:
    print(f"  ❌ {'pymupdf (fitz)':<32} {str(e)[:48]}")
total += 1

try:
    from imwatermark import WatermarkEncoder  # noqa: F401
    print(f"  ✅ {'invisible_watermark':<32} (installed)"); passed += 1
except Exception as e:
    print(f"  ❌ {'invisible_watermark':<32} {str(e)[:48]}")
total += 1

try:
    from PIL import Image  # noqa: F401
    print(f"  ✅ {'PIL (pillow)':<32} (installed)"); passed += 1
except Exception as e:
    print(f"  ❌ {'PIL (pillow)':<32} {str(e)[:48]}")
total += 1

try:
    import cv2
    print(f"  ✅ {'cv2 (opencv)':<32} {cv2.__version__}"); passed += 1
except Exception as e:
    print(f"  ❌ {'cv2 (opencv)':<32} {str(e)[:48]}")
total += 1

# ── 4.2.4 Audio & Voice Processing ───────────────────────────────────────────
print("\n  4.2.4 — Audio & Voice Processing")
print("  " + "─" * 56)

try:
    from transformers import pipeline as _hf_pipeline  # noqa: F401
    # voice STT uses transformers.pipeline("automatic-speech-recognition")
    print(f"  ✅ {'transformers pipeline (voice STT)':<32} (available)"); passed += 1
except Exception as e:
    print(f"  ❌ {'transformers pipeline (voice STT)':<32} {str(e)[:48]}")
total += 1

try:
    import torchaudio
    print(f"  ✅ {'torchaudio':<32} {torchaudio.__version__}"); passed += 1
except Exception as e:
    print(f"  ❌ {'torchaudio':<32} {str(e)[:48]}")
total += 1

try:
    import librosa
    print(f"  ✅ {'librosa':<32} {librosa.__version__}"); passed += 1
except Exception as e:
    print(f"  ❌ {'librosa':<32} {str(e)[:48]}")
total += 1

try:
    import soundfile
    print(f"  ✅ {'soundfile':<32} {soundfile.__version__}"); passed += 1
except Exception as e:
    print(f"  ❌ {'soundfile':<32} {str(e)[:48]}")
total += 1

try:
    import sounddevice
    print(f"  ✅ {'sounddevice':<32} {sounddevice.__version__}"); passed += 1
except Exception as e:
    print(f"  ❌ {'sounddevice':<32} {str(e)[:48]}")
total += 1

# ── 4.2.5 Agentic AI & RAG ───────────────────────────────────────────────────
print("\n  4.2.5 — Agentic AI & RAG")
print("  " + "─" * 56)

try:
    import langchain
    print(f"  ✅ {'langchain':<32} {langchain.__version__}"); passed += 1
except Exception as e:
    print(f"  ❌ {'langchain':<32} {str(e)[:48]}")
total += 1

try:
    from langchain_community.llms import LlamaCpp  # noqa: F401
    print(f"  ✅ {'langchain_community (LlamaCpp)':<32} (installed)"); passed += 1
except Exception as e:
    print(f"  ❌ {'langchain_community (LlamaCpp)':<32} {str(e)[:48]}")
total += 1

try:
    import chromadb
    print(f"  ✅ {'chromadb':<32} {chromadb.__version__}"); passed += 1
except Exception as e:
    print(f"  ❌ {'chromadb':<32} {str(e)[:48]}")
total += 1

try:
    import sentence_transformers
    print(f"  ✅ {'sentence_transformers':<32} {sentence_transformers.__version__}"); passed += 1
except Exception as e:
    print(f"  ❌ {'sentence_transformers':<32} {str(e)[:48]}")
total += 1

# ── 4.2.6 Monitoring & Utilities ─────────────────────────────────────────────
print("\n  4.2.6 — Monitoring & Utilities")
print("  " + "─" * 56)

try:
    import psutil
    print(f"  ✅ {'psutil':<32} {psutil.__version__}"); passed += 1
except Exception as e:
    print(f"  ❌ {'psutil':<32} {str(e)[:48]}")
total += 1

try:
    import GPUtil  # noqa: F401
    print(f"  ✅ {'gputil':<32} (installed)"); passed += 1
except Exception as e:
    print(f"  ❌ {'gputil':<32} {str(e)[:48]}")
total += 1

try:
    import plotly
    print(f"  ✅ {'plotly':<32} {plotly.__version__}"); passed += 1
except Exception as e:
    print(f"  ❌ {'plotly':<32} {str(e)[:48]}")
total += 1

try:
    import yaml
    print(f"  ✅ {'pyyaml':<32} {yaml.__version__}"); passed += 1
except Exception as e:
    print(f"  ❌ {'pyyaml':<32} {str(e)[:48]}")
total += 1

try:
    from dotenv import load_dotenv  # noqa: F401
    print(f"  ✅ {'python-dotenv':<32} (installed)"); passed += 1
except Exception as e:
    print(f"  ❌ {'python-dotenv':<32} {str(e)[:48]}")
total += 1

# ── FAISS — GPU or CPU both acceptable ───────────────────────────────────────
print("\n  FAISS")
print("  " + "─" * 56)
total += 1
try:
    import faiss
    _is_gpu = hasattr(faiss, "StandardGpuResources")
    passed += 1
    if _is_gpu:
        print("  ✅ faiss-gpu-cu12                GPU-accelerated")
    else:
        print("  ✅ faiss-cpu                     CPU fallback (functional for this project)")
except Exception as e:
    print(f"  ❌ faiss                          {str(e)[:48]}")
    print("       Re-run Cell 6 to retry the faiss install.")

# ── CoquiTTS — import + version check ────────────────────────────────────────
print("\n  CoquiTTS / XTTS v2 (voice TTS)")
print("  " + "─" * 56)
total += 1
try:
    import TTS as _tts_pkg
    from TTS.api import TTS as _TTS  # noqa: F401
    passed += 1
    print(f"  ✅ {'TTS (CoquiTTS)':<32} {_tts_pkg.__version__}")
except Exception as e:
    print(f"  ❌ {'TTS (CoquiTTS)':<32} {str(e)[:56]}")
    print("       Re-run Cell 6 to reinstall coqui-tts[codec].")

# ── pyaudio — non-blocking ───────────────────────────────────────────────────
print("\n  pyaudio (real-time microphone capture)")
print("  " + "─" * 56)
try:
    import pyaudio  # noqa: F401
    print("  ✅ pyaudio                       (installed)")
except ImportError:
    print("  ⚠️  pyaudio                       not installed  [non-blocking]")
    print("       No starter notebook requires real-time mic capture (voice demo uses file upload).")

print("\n" + "═" * 60)
print(f"  Grand Total: {passed}/{total} libraries available")
print("═" * 60)

if passed == total:
    print("✅ Cell 7 complete — all libraries verified")
else:
    failed = total - passed
    print(f"⚠️  {failed} library/libraries not available — re-run Cell 6 if unexpected")
    print("   Note: faiss-cpu or pyaudio failures do NOT block any starter notebook.")

════════════════════════════════════════════════════════════
  Infrastructure Test — AI Libraries
════════════════════════════════════════════════════════════

  4.2.1 — Core ML/AI Frameworks
  ────────────────────────────────────────────────────────
  ✅ transformers                     4.57.6
  ✅ diffusers                        0.36.0
  ✅ accelerate                       1.12.0
  ✅ bitsandbytes                     0.49.2
  ✅ safetensors                      0.7.0
  ✅ sentencepiece                    0.2.1
  ✅ vllm                             0.15.1
  ✅ xformers                         0.0.33.post2

  4.2.2 — Interface & Deployment
  ────────────────────────────────────────────────────────
  ✅ mlflow                           3.1.0
  ✅ huggingface_hub                  0.36.2
  ✅ fastapi                          0.128.0
  ✅ pydantic                         2.12.5
  ✅ datasets                         3.0.1
  ✅ streamlit                        1.54.0

  4.2.3 — Document & Media Processin

2026-03-03 16:57:25.194721981 [W:onnxruntime:Default, device_discovery.cc:164 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:89 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


  ✅ sentence_transformers            3.0.0

  4.2.6 — Monitoring & Utilities
  ────────────────────────────────────────────────────────
  ✅ psutil                           7.2.1
  ✅ gputil                           (installed)
  ✅ plotly                           6.5.2
  ✅ pyyaml                           6.0.3
  ✅ python-dotenv                    (installed)

  FAISS
  ────────────────────────────────────────────────────────
  ✅ faiss-gpu-cu12                GPU-accelerated

  CoquiTTS / XTTS v2 (voice TTS)
  ────────────────────────────────────────────────────────
  ❌ TTS (CoquiTTS)                   Numba needs NumPy 2.2 or less. Got NumPy 2.4.
       Re-run Cell 6 to reinstall coqui-tts[codec].

  pyaudio (real-time microphone capture)
  ────────────────────────────────────────────────────────
  ✅ pyaudio                       (installed)

════════════════════════════════════════════════════════════
  Grand Total: 34/35 libraries available
═════════════════════════════════════════

## 8. Hugging Face Authentication

Some models in this project require a **free Hugging Face account** and an **access token**.

### Steps to get your token:
1. Create a free account at [huggingface.co](https://huggingface.co/join)
2. Go to **Settings → Access Tokens** → [https://huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)
3. Click **"New token"** → Name it (e.g., `ai-studio`) → select **"Read"** → **"Generate token"**
4. Copy the token — it starts with `hf_...`

### For FLUX.1-dev (gated model):
FLUX.1-dev requires accepting the license before download:
5. Visit [https://huggingface.co/black-forest-labs/FLUX.1-dev](https://huggingface.co/black-forest-labs/FLUX.1-dev)
6. Click **"Access repository"** and accept the license agreement
7. Wait a few minutes for your access to be approved (usually instant)

> **Security note:** Your token is entered via `getpass` — it will not appear on screen
> and will not be saved to the notebook file. After first login, the authenticated token will be saved on cache to avoid the token prompt every time this notebook is executed.

In [9]:

import getpass
from huggingface_hub import login, whoami

# Check if already authenticated
hf_auth_ok = False
try:
    user = whoami()
    print(f"✅ Already authenticated as : {user['name']}")
    print(f"   Account type             : {user.get('type', 'user')}")
    print(f"   Email                    : {user.get('email', '(not set)')}")
    hf_auth_ok = True
except Exception:
    hf_token = getpass.getpass("🔑 Paste your Hugging Face token (hidden): ")
    try:
        # login() caches the token in ~/.cache/huggingface/token
        # add_to_git_credential=False keeps it out of git credential storage
        login(token=hf_token, add_to_git_credential=False)
        user = whoami()
        print(f"✅ Authenticated as : {user['name']}")
        print(f"   Account type     : {user.get('type', 'user')}")
        print(f"   Email            : {user.get('email', '(not set)')}")
        hf_auth_ok = True
    except Exception as e:
        print(f"❌ Authentication failed: {e}")
        print("   Check your token at https://huggingface.co/settings/tokens")
        hf_auth_ok = False

print()
print("✅ Cell 8 complete" if hf_auth_ok else "⚠️  Fix auth before model downloads")


✅ Already authenticated as : dcruas
   Account type             : user
   Email                    : dcruas@yahoo.com.br

✅ Cell 8 complete


## 9. Model Download

Download all quantized GGUF models used by the four starter notebooks.
Each download is **skipped automatically** if the file already exists — re-running this
cell is safe and will only fetch what is missing.

<div class="alert alert-block alert-danger">
<b>🔒 FLUX.1-dev is a gated model — license acceptance required BEFORE running this cell.</b>
<br><br>
FLUX.1-dev is hosted on Hugging Face under a restricted license. The download cell below will
fail with a <code>401 Unauthorized</code> error for FLUX if you have not completed both steps:
<ol>
  <li>
    Visit <a href="https://huggingface.co/black-forest-labs/FLUX.1-dev" target="_blank">
    <b>huggingface.co/black-forest-labs/FLUX.1-dev</b></a>, click <b>"Access repository"</b>,
    and accept the license agreement.
    Access is usually granted within a few minutes.
  </li>
  <li>
    Complete <b>Cell 8 (Hugging Face Auth)</b> so your authenticated token is on file.
  </li>
</ol>
<b>Note:</b> Skipping these steps only breaks the FLUX download.
All other models (Zephyr, Llama 3.1, Whisper, XTTS v2) download without any license gate.
</div>

> ⚠️ **Total download size:** ~40 GB. Ensure you have sufficient disk space at
> `/home/jovyan/local/` before proceeding.

| # | Capability | Model | Repo | Size |
|---|-----------|-------|------|------|
| 1 | Chatbot | Zephyr 7B Beta Q5_K_M | `TheBloke/zephyr-7B-beta-GGUF` | ~4.8 GB |
| 2 | Document + Voice LLM | Llama 3.1 8B Q6_K_L | `bartowski/Meta-Llama-3.1-8B-Instruct-GGUF` | ~6.6 GB |
| 3a | Image Gen (GGUF transformer) | FLUX.1-dev Q4_K_S | `city96/FLUX.1-dev-gguf` | ~6.9 GB |
| 3b | Image Gen (pipeline components) | FLUX.1-dev encoders + VAE | `black-forest-labs/FLUX.1-dev` | ~22 GB |
| 4 | Voice STT | Whisper Large V3 Turbo | `openai/whisper-large-v3-turbo` | ~1.6 GB |
| 5 | Voice TTS | XTTS v2 (CoquiTTS native) | `coqui/XTTS-v2` | ~1.8 GB |

In [10]:

import os
import time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from huggingface_hub import hf_hub_download, snapshot_download

# ── Base directory for all locally stored models ──────────────────────────────
LOCAL_BASE = Path("/home/jovyan/local")
LOCAL_BASE.mkdir(parents=True, exist_ok=True)


def _skip_or_download(label: str, check_path: Path, download_fn) -> tuple[str, bool, str]:
    """Download a model asset unless it already exists at check_path.
    Returns (label, success, path_str).
    """
    if check_path.exists():
        size_gb = (
            sum(f.stat().st_size for f in check_path.rglob("*") if f.is_file()) / 1e9
            if check_path.is_dir()
            else check_path.stat().st_size / 1e9
        )
        print(f"  ⏭️  {label} — already exists ({size_gb:.2f} GB)")
        return label, True, str(check_path)
    print(f"  ⬇️  {label} — starting download ...")
    t0 = time.time()
    try:
        download_fn()
        elapsed = time.time() - t0
        size_gb = (
            sum(f.stat().st_size for f in check_path.rglob("*") if f.is_file()) / 1e9
            if check_path.is_dir()
            else check_path.stat().st_size / 1e9
        )
        print(f"  ✅ {label} — done in {elapsed:.0f}s ({size_gb:.2f} GB)")
        return label, True, str(check_path)
    except Exception as e:
        print(f"  ❌ {label} — FAILED: {e}")
        return label, False, str(check_path)


print("─" * 60)
print("  Model Download  —  /home/jovyan/local/  (parallel)")
print("─" * 60)

# Pre-create destination directories
for sub in ["zephyr-7b-beta", "meta-llama3.1-8b-Q6", "flux1-dev",
            "whisper-large-v3-turbo", "xtts-v2"]:
    (LOCAL_BASE / sub).mkdir(parents=True, exist_ok=True)
(LOCAL_BASE / "flux1-dev" / "transformer").mkdir(parents=True, exist_ok=True)

# ── Define all download tasks ─────────────────────────────────────────────────
tasks = [
    # (label, check_path, download_fn, result_path_for_summary)
    (
        "Chatbot LLM  — Zephyr 7B Beta Q5_K_M  (~4.8 GB)",
        LOCAL_BASE / "zephyr-7b-beta" / "zephyr-7b-beta.Q5_K_M.gguf",
        lambda: hf_hub_download(
            repo_id="TheBloke/zephyr-7B-beta-GGUF",
            filename="zephyr-7b-beta.Q5_K_M.gguf",
            local_dir=str(LOCAL_BASE / "zephyr-7b-beta"),
        ),
        "Chatbot LLM  (Zephyr 7B Beta Q5_K_M)",
    ),
    (
        "Document + Voice LLM  — Llama 3.1 8B Q6_K_L  (~6.6 GB)",
        LOCAL_BASE / "meta-llama3.1-8b-Q6" / "Meta-Llama-3.1-8B-Instruct-Q6_K_L.gguf",
        lambda: hf_hub_download(
            repo_id="bartowski/Meta-Llama-3.1-8B-Instruct-GGUF",
            filename="Meta-Llama-3.1-8B-Instruct-Q6_K_L.gguf",
            local_dir=str(LOCAL_BASE / "meta-llama3.1-8b-Q6"),
        ),
        "Document + Voice LLM  (Llama 3.1 8B Q6_K_L)",
    ),
    (
        "Image Gen GGUF transformer  — FLUX.1-dev Q4_K_S  (~6.9 GB)",
        LOCAL_BASE / "flux1-dev" / "flux1-dev-Q4_K_S.gguf",
        lambda: hf_hub_download(
            repo_id="city96/FLUX.1-dev-gguf",
            filename="flux1-dev-Q4_K_S.gguf",
            local_dir=str(LOCAL_BASE / "flux1-dev"),
        ),
        "Image Gen GGUF transformer  (FLUX.1-dev Q4_K_S)",
    ),
    (
        "Image Gen pipeline components  — FLUX.1-dev encoders + VAE  (~22 GB)",
        LOCAL_BASE / "flux1-dev" / "model_index.json",
        lambda: snapshot_download(
            repo_id="black-forest-labs/FLUX.1-dev",
            local_dir=str(LOCAL_BASE / "flux1-dev"),
            ignore_patterns=["transformer/*", "*.bin"],
        ),
        "Image Gen pipeline  (FLUX.1-dev encoders + VAE)",
    ),
    (
        "Image Gen transformer config  — FLUX.1-dev config.json  (~1 KB)",
        LOCAL_BASE / "flux1-dev" / "transformer" / "config.json",
        lambda: hf_hub_download(
            repo_id="black-forest-labs/FLUX.1-dev",
            filename="transformer/config.json",
            local_dir=str(LOCAL_BASE / "flux1-dev"),
        ),
        "Image Gen transformer config  (FLUX.1-dev config.json)",
    ),
    (
        "Voice STT  — Whisper Large V3 Turbo  (~1.6 GB)",
        LOCAL_BASE / "whisper-large-v3-turbo" / "config.json",
        lambda: snapshot_download(
            repo_id="openai/whisper-large-v3-turbo",
            local_dir=str(LOCAL_BASE / "whisper-large-v3-turbo"),
        ),
        "Voice STT  (Whisper Large V3 Turbo HuggingFace)",
    ),
    (
        # CoquiTTS uses its own native PyTorch format (.pth), NOT GGUF.
        # snapshot_download fetches model.pth, config.json, vocab.json,
        # speakers_xtts.pth, and dvae.pth — everything CoquiTTS needs to
        # load the model locally via TTS(model_path=..., config_path=...).
        # The whole directory is bundled as an MLflow artifact at registration
        # time so the serving container never needs to download it at runtime.
        "Voice TTS  — XTTS v2 (CoquiTTS native)  (~1.8 GB)",
        LOCAL_BASE / "xtts-v2" / "model.pth",
        lambda: snapshot_download(
            repo_id="coqui/XTTS-v2",
            local_dir=str(LOCAL_BASE / "xtts-v2"),
        ),
        "Voice TTS  (XTTS v2 CoquiTTS native)",
    ),
]

# ── Run downloads in parallel (up to 8 concurrent) ───────────────────────────
t_start = time.time()
download_results = []

with ThreadPoolExecutor(max_workers=8) as pool:
    futures = {
        pool.submit(_skip_or_download, label, check_path, fn): summary_label
        for label, check_path, fn, summary_label in tasks
    }
    for future in as_completed(futures):
        _label, ok, path = future.result()
        summary_label = futures[future]
        download_results.append((summary_label, ok, path))

# ── Summary ───────────────────────────────────────────────────────────────────
elapsed_total = time.time() - t_start
print()
print("─" * 60)
print("  Download Summary")
print("─" * 60)
all_ok = True
for label, status, path in download_results:
    icon = "✅" if status else "❌"
    print(f"  {icon} {label}")
    print(f"       {path}")
    if not status:
        all_ok = False
print("─" * 60)
print(f"  Total download time: {elapsed_total:.0f}s  ({elapsed_total / 60:.1f} min)")
print("─" * 60)
if all_ok:
    print("✅ Cell 9 complete — all models ready")
else:
    print("⚠️  Some downloads failed. Check errors above and re-run.")


────────────────────────────────────────────────────────────
  Model Download  —  /home/jovyan/local/  (parallel)
────────────────────────────────────────────────────────────
  ⏭️  Document + Voice LLM  — Llama 3.1 8B Q6_K_L  (~6.6 GB) — already exists (6.85 GB)
  ⏭️  Chatbot LLM  — Zephyr 7B Beta Q5_K_M  (~4.8 GB) — already exists (5.13 GB)
  ⏭️  Image Gen pipeline components  — FLUX.1-dev encoders + VAE  (~22 GB) — already exists (0.00 GB)
  ⏭️  Image Gen GGUF transformer  — FLUX.1-dev Q4_K_S  (~6.9 GB) — already exists (6.81 GB)
  ⏭️  Voice STT  — Whisper Large V3 Turbo  (~1.6 GB) — already exists (0.00 GB)
  ⏭️  Image Gen transformer config  — FLUX.1-dev config.json  (~1 KB) — already exists (0.00 GB)
  ⏭️  Voice TTS  — XTTS v2 (CoquiTTS native)  (~1.8 GB) — already exists (1.87 GB)

────────────────────────────────────────────────────────────
  Download Summary
────────────────────────────────────────────────────────────
  ✅ Document + Voice LLM  (Llama 3.1 8B Q6_K_L)
       /home

## 10. Setup Summary

Full status report of the environment after completing all cells above.

In [11]:
import gc
import torch
from pathlib import Path

LOCAL_BASE = Path("/home/jovyan/local")

# ── GPU ───────────────────────────────────────────────────────────────────────
print("─" * 60)
print("  GPU Status")
print("─" * 60)
if torch.cuda.is_available():
    props  = torch.cuda.get_device_properties(0)
    free_b, total_b = torch.cuda.mem_get_info()
    print(f"  GPU      : {props.name}")
    print(f"  VRAM     : {props.total_memory / 1e9:.1f} GB total  |  {free_b / 1e9:.1f} GB free")
    print(f"  Compute  : {props.major}.{props.minor}")
    print(f"  Driver   : CUDA {torch.version.cuda}")
else:
    print("  ❌ GPU not available")

# ── HF Auth ───────────────────────────────────────────────────────────────────
print()
print("─" * 60)
print("  Hugging Face Auth")
print("─" * 60)
try:
    from huggingface_hub import whoami
    user = whoami()
    print(f"  ✅ Logged in as: {user['name']}")
except Exception:
    print("  ❌ Not authenticated — re-run Cell 8")

# ── Model Files ───────────────────────────────────────────────────────────────
print()
print("─" * 60)
print("  Model File Status")
print("─" * 60)
_model_checks = [
    ("Chatbot LLM  Zephyr 7B Beta Q5_K_M",
     LOCAL_BASE / "zephyr-7b-beta" / "zephyr-7b-beta.Q5_K_M.gguf"),
    ("Document LLM  Llama 3.1 8B Q6_K_L",
     LOCAL_BASE / "meta-llama3.1-8b-Q6" / "Meta-Llama-3.1-8B-Instruct-Q6_K_L.gguf"),
    ("Voice LLM  Llama 3.1 8B Q6_K_L  (shared)",
     LOCAL_BASE / "meta-llama3.1-8b-Q6" / "Meta-Llama-3.1-8B-Instruct-Q6_K_L.gguf"),
    ("Image Gen  FLUX.1-dev GGUF transformer",
     LOCAL_BASE / "flux1-dev" / "flux1-dev-Q4_K_S.gguf"),
    ("Image Gen  FLUX.1-dev pipeline",
     LOCAL_BASE / "flux1-dev" / "model_index.json"),
    ("Voice STT  Whisper Large V3 Turbo",
     LOCAL_BASE / "whisper-large-v3-turbo" / "config.json"),
    # CoquiTTS native format — model.pth is the main weight file.
    # voice.yaml tts_model_path points to the parent directory, which is
    # bundled as an artifact at registration time.
    ("Voice TTS  XTTS v2 (CoquiTTS native)",
     LOCAL_BASE / "xtts-v2" / "model.pth"),
]
all_present = True
for label, path in _model_checks:
    exists = path.exists()
    icon   = "✅" if exists else "❌"
    size   = f"({path.stat().st_size / 1e9:.2f} GB)" if exists and path.is_file() else ""
    print(f"  {icon} {label} {size}")
    print(f"       {path}")
    if not exists:
        all_present = False

# ── Next Steps ────────────────────────────────────────────────────────────────
elapsed = time.time() - start_time
print()
print("─" * 60)
print(f"  Total setup time: {elapsed:.0f}s  ({elapsed / 60:.1f} min)")
print("─" * 60)
print()
if all_present:
    print("🎉 Setup complete! Open a starter notebook to begin:")
    print()
    print("  📂 chatbot-starter.ipynb          ← Conversational AI  (Zephyr 7B)")
    print("  📂 document-analyzer-starter.ipynb ← Document Q&A  (Llama 3.1 8B)")
    print("  📂 image-gen-starter.ipynb         ← Text-to-Image  (FLUX.1-dev)")
    print("  📂 voice-assistant-starter.ipynb   ← Voice AI  (Whisper + XTTS v2)")
else:
    print("⚠️  Some models are still missing — re-run Cell 9 to retry downloads.")


────────────────────────────────────────────────────────────
  GPU Status
────────────────────────────────────────────────────────────
  GPU      : NVIDIA RTX PRO 5000 Blackwell Generation Laptop GPU
  VRAM     : 25.7 GB total  |  24.2 GB free
  Compute  : 12.0
  Driver   : CUDA 12.8

────────────────────────────────────────────────────────────
  Hugging Face Auth
────────────────────────────────────────────────────────────
  ✅ Logged in as: dcruas

────────────────────────────────────────────────────────────
  Model File Status
────────────────────────────────────────────────────────────
  ✅ Chatbot LLM  Zephyr 7B Beta Q5_K_M (5.13 GB)
       /home/jovyan/local/zephyr-7b-beta/zephyr-7b-beta.Q5_K_M.gguf
  ✅ Document LLM  Llama 3.1 8B Q6_K_L (6.85 GB)
       /home/jovyan/local/meta-llama3.1-8b-Q6/Meta-Llama-3.1-8B-Instruct-Q6_K_L.gguf
  ✅ Voice LLM  Llama 3.1 8B Q6_K_L  (shared) (6.85 GB)
       /home/jovyan/local/meta-llama3.1-8b-Q6/Meta-Llama-3.1-8B-Instruct-Q6_K_L.gguf
  ✅ Image Gen 

## 11. Quick Reference

Common code snippets for working with the downloaded models in the starter notebooks.

### Chatbot — Zephyr 7B Beta (LlamaCpp)
```python
from langchain_community.llms import LlamaCpp

llm = LlamaCpp(
    model_path="/home/jovyan/local/zephyr-7b-beta/zephyr-7b-beta.Q5_K_M.gguf",
    n_gpu_layers=-1, n_ctx=4096, temperature=0.7,
)

# Zephyr prompt template (ChatML-like)
prompt = """<|system|>
You are a helpful assistant.</s>
<|user|>
What is the Eiffel Tower?</s>
<|assistant|>
"""
response = llm(prompt)
```

### Document Analyzer + Voice LLM — Llama 3.1 8B (LlamaCpp)
```python
from langchain_community.llms import LlamaCpp

llm = LlamaCpp(
    model_path="/home/jovyan/local/meta-llama3.1-8b-Q6/Meta-Llama-3.1-8B-Instruct-Q6_K_L.gguf",
    n_gpu_layers=-1, n_ctx=8192, temperature=0.0,
)

# Llama 3.1 prompt template
prompt = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a helpful assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>
Summarise this text.<|eot_id|><|start_header_id|>assistant<|end_header_id|>"""
```

### Image Generation — FLUX.1-dev GGUF (diffusers)
```python
import torch
from diffusers import FluxPipeline, FluxTransformer2DModel
from diffusers.utils import GGUFQuantizationConfig

MODEL_DIR = "/home/jovyan/local/flux1-dev"
transformer = FluxTransformer2DModel.from_single_file(
    f"{MODEL_DIR}/flux1-dev-Q4_K_S.gguf",
    quantization_config=GGUFQuantizationConfig(compute_dtype=torch.bfloat16),
    torch_dtype=torch.bfloat16,
)
pipe = FluxPipeline.from_pretrained(MODEL_DIR, transformer=transformer, torch_dtype=torch.bfloat16)
pipe.enable_model_cpu_offload()
image = pipe("A red cat on a spaceship", num_inference_steps=28, guidance_scale=3.5).images[0]
```

### Voice STT — Whisper Large V3 Turbo (transformers)
```python
import torch
from transformers import pipeline as hf_pipeline

stt = hf_pipeline(
    "automatic-speech-recognition",
    model="/home/jovyan/local/whisper-large-v3-turbo",
    device=0 if torch.cuda.is_available() else -1,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
)
result = stt(audio_array_16khz, generate_kwargs={"language": "english"})
transcription = result["text"].strip()
```

### Voice TTS — XTTS v2 (TTS / CoquiTTS)
```python
from TTS.api import TTS

tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to("cuda")
tts.tts_to_file(text="Hello from AI Studio!", speaker_wav="ref.wav",
                language="en", file_path="output.wav")
```

### Common Error Fixes
| Error | Fix |

|-------|-----|| `Import error` | Re-run Cell 6 (AI Library Install) then kernel restart |

| `CUDA out of memory` | `torch.cuda.empty_cache(); import gc; gc.collect()` || `FLUX access denied` | Accept license at huggingface.co/black-forest-labs/FLUX.1-dev |

| `Model not found` | Re-run Cell 9 (Model Download) || `401 Unauthorized` | Re-run Cell 8 (HF Auth) |